# SVM Multi-Class Classification — Purkinje Cell Lick Dataset

This notebook trains a **Support Vector Machine (SVM)** to classify lick types from Purkinje cell spike data.

## Label Mapping

| Label | Meaning |
|-------|------------------------------------------|
| 0     | Grooming (bout_tag == 0) |
| 1     | Left lick |
| 2     | Right lick |
| 3     | Left miss |
| 4     | Right miss |
| 5     | Left lick + reward consumed |
| 6     | Right lick + reward consumed |
| 7     | Inter-bout lick |

## Features extracted per lick (per electrode)
- **SS firing rate** in the 100ms pre-lick window (Hz)
- **CS count** in the 100ms pre-lick window
- **Velocity max** (`tongue_vm_max`)
- **Deceleration min** (`tongue_vm_min`)
- **Deceleration strength** (Δv / Δt)

## Pipeline
1. Re-run alignment on both `.mat` files
2. Engineer per-lick feature vectors
3. Train/test split
4. Scale features
5. Train SVM (RBF kernel) with `GridSearchCV`
6. Evaluate: accuracy, confusion matrix, per-class F1


## 1 — Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

print('All imports OK')
print(f'scikit-learn version: {__import__("sklearn").__version__}')

## 2 — Configuration

In [ ]:
# ── File paths ────────────────────────────────────────────────────────────────
MAT_FILES = ['1D_1.mat', '1D_2.mat']

# ── Alignment window ─────────────────────────────────────────────────────────
WINDOW_PRE = 0.100   # seconds before lick onset

# ── Label names (for display) ─────────────────────────────────────────────────
LABEL_NAMES = {
    0: 'Grooming',
    1: 'Left lick',
    2: 'Right lick',
    3: 'Left miss',
    4: 'Right miss',
    5: 'Left+reward',
    6: 'Right+reward',
    7: 'Inter-bout',
}

# ── Random seed ───────────────────────────────────────────────────────────────
SEED = 42

print('Configuration loaded.')

## 3 — Data Loading & Alignment

We loop over both `.mat` files, resolve MATLAB cell-array references, and align neural spikes to each lick onset.

In [ ]:
def load_and_align(filepath: str, window_pre: float = 0.100) -> list:
    """
    Load a .mat file and return a list of per-lick dicts, each containing:
      - lick_id, source_file, final_label
      - velocity_max, deceleration_min, decel_strength
      - spike_times: list of {SS: array, CS: array} per electrode
    """
    aligned = []

    with h5py.File(filepath, 'r') as f:
        # ── Neural references ─────────────────────────────────────────────────
        neural_refs = f['data_recordings']['Neural_Data'][()].flatten()

        # ── Tongue / lick references ──────────────────────────────────────────
        tongue_ref = f['data_recordings']['tongue'][()].flatten()[0]
        lick_group = f[tongue_ref]

        raw_lick_tags  = lick_group['tag_lick'][()].flatten()
        raw_bout_tags  = lick_group['tag_bout'][()].flatten()
        harvest_tags   = lick_group['tag_harvest'][()].flatten()
        all_onsets     = lick_group['time_onset'][()].flatten()
        velocities     = lick_group['tongue_vm_max'][()].flatten()
        decelerations  = lick_group['tongue_vm_min'][()].flatten()
        time_vmax      = lick_group['time_vmax'][()].flatten()
        time_vmin      = lick_group['time_vmin'][()].flatten()

        # ── Build final labels (grooming override) ────────────────────────────
        final_labels = [
            0 if raw_bout_tags[i] == 0 else int(raw_lick_tags[i])
            for i in range(len(raw_lick_tags))
        ]

        # ── Per-lick alignment ────────────────────────────────────────────────
        for i in range(len(all_onsets)):
            t_onset  = all_onsets[i]
            t_start  = t_onset - window_pre
            t_end    = t_onset

            lick_spikes = []
            for ref in neural_refs:
                ss_all = f[ref]['SS_time'][()].flatten()
                cs_all = f[ref]['CS_time'][()].flatten()

                ss_rel = ss_all[(ss_all >= t_start) & (ss_all < t_end)] - t_end
                cs_rel = cs_all[(cs_all >= t_start) & (cs_all < t_end)] - t_end

                lick_spikes.append({'SS': ss_rel, 'CS': cs_rel})

            # ── Deceleration strength ─────────────────────────────────────────
            dt = time_vmin[i] - time_vmax[i]
            decel_strength = velocities[i] / dt if dt > 0 else np.nan

            aligned.append({
                'lick_id':          i,
                'source_file':      filepath,
                'final_label':      final_labels[i],
                'bout_tag':         raw_bout_tags[i],
                'harvest_tag':      harvest_tags[i],
                'velocity_max':     velocities[i],
                'deceleration_min': decelerations[i],
                'decel_strength':   decel_strength,
                'spike_times':      lick_spikes,
            })

    print(f'  {filepath}: {len(aligned)} licks, {len(neural_refs)} electrodes')
    return aligned


# ── Load both files ───────────────────────────────────────────────────────────
all_licks = []
print('Loading files...')
for f in MAT_FILES:
    try:
        all_licks.extend(load_and_align(f, WINDOW_PRE))
    except FileNotFoundError:
        print(f'  WARNING: {f} not found — skipping.')

print(f'\nTotal licks loaded: {len(all_licks)}')

## 4 — Feature Engineering

For each lick we compute a **fixed-length feature vector**:

| Feature | Description |
|---------|-------------|
| `ss_rate_e{k}` | SS firing rate (Hz) in 100ms window, electrode *k* |
| `cs_count_e{k}` | CS spike count in window, electrode *k* |
| `velocity_max` | Peak tongue protraction velocity |
| `deceleration_min` | Peak tongue retraction velocity |
| `decel_strength` | Δv / Δt proxy for deceleration |

All features are concatenated into one row per lick.

In [ ]:
def build_feature_matrix(lick_list: list) -> pd.DataFrame:
    """
    Convert list-of-lick-dicts → tidy feature DataFrame.
    One row per lick.  NaN-safe: missing kinematic values are filled with 0.
    """
    rows = []
    for lick in lick_list:
        row = {
            'lick_id':     lick['lick_id'],
            'source_file': lick['source_file'],
            'label':       lick['final_label'],
        }

        # ── Per-electrode spike features ──────────────────────────────────────
        for e_idx, spk in enumerate(lick['spike_times']):
            row[f'ss_rate_e{e_idx}']  = len(spk['SS']) / WINDOW_PRE   # Hz
            row[f'cs_count_e{e_idx}'] = len(spk['CS'])                # count

        # ── Kinematic features ────────────────────────────────────────────────
        row['velocity_max']     = lick['velocity_max']
        row['deceleration_min'] = lick['deceleration_min']
        row['decel_strength']   = lick['decel_strength']

        rows.append(row)

    df = pd.DataFrame(rows)
    return df


feat_df = build_feature_matrix(all_licks)

# ── Quick audit ───────────────────────────────────────────────────────────────
feature_cols = [c for c in feat_df.columns if c not in ('lick_id', 'source_file', 'label')]
print(f'Feature matrix shape : {feat_df.shape}')
print(f'Number of features   : {len(feature_cols)}')
print(f'Features             : {feature_cols}')
print(f'\nMissing values:\n{feat_df[feature_cols].isnull().sum()[feat_df[feature_cols].isnull().sum() > 0]}')
feat_df.head(3)

In [ ]:
# ── Handle NaNs in decel_strength (divide-by-zero edge case) ─────────────────
feat_df['decel_strength'] = feat_df['decel_strength'].fillna(0.0)

# ── Label distribution ────────────────────────────────────────────────────────
label_counts = feat_df['label'].value_counts().sort_index()
label_names_list = [LABEL_NAMES.get(int(l), str(l)) for l in label_counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(label_counts)), label_counts.values,
              color=sns.color_palette('tab10', len(label_counts)))
ax.set_xticks(range(len(label_counts)))
ax.set_xticklabels([f'{int(k)}\n{n}' for k, n in zip(label_counts.index, label_names_list)],
                   fontsize=9)
ax.set_title('Label Distribution Across Both Files', fontsize=13, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Label ID / Name')
for bar, val in zip(bars, label_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

print('\nLabel counts:')
for k, v in zip(label_counts.index, label_counts.values):
    print(f'  Label {int(k):2d}  {LABEL_NAMES.get(int(k), "??"):15s}  n={v}')

## 5 — Prepare X and y

> **Note on class imbalance:** Labels 0 (Grooming) and 7 (Inter-bout) dominate the dataset.  
> We pass `class_weight='balanced'` to the SVM so that minority classes are not ignored.

In [ ]:
X = feat_df[feature_cols].values.astype(np.float64)
y = feat_df['label'].values.astype(int)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Classes : {sorted(np.unique(y))}')

## 6 — Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y          # preserve label proportions in both sets
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'\nTrain label counts: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  label counts: {dict(zip(*np.unique(y_test,  return_counts=True)))}')

## 7 — SVM Pipeline (Scaler + SVC)

We bundle `StandardScaler` and `SVC` into a single `Pipeline` so that the scaler is always fitted only on training data — no data leakage.

In [ ]:
# ── Build pipeline ────────────────────────────────────────────────────────────
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svc',    SVC(
        kernel='rbf',
        class_weight='balanced',   # handles imbalanced labels
        random_state=SEED,
        probability=True           # needed for predict_proba later if required
    ))
])

# ── Hyperparameter grid ───────────────────────────────────────────────────────
param_grid = {
    'svc__C':     [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.01, 0.001],
}

# ── Cross-validation strategy ─────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Pipeline:')
print(svm_pipeline)
print(f'\nParam grid : {param_grid}')
print(f'CV strategy: StratifiedKFold(n_splits=5)')

In [ ]:
# ── Grid search ───────────────────────────────────────────────────────────────
grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='balanced_accuracy',   # appropriate for imbalanced classes
    n_jobs=-1,
    verbose=1,
    refit=True                     # refit best model on full training set
)

print('Running GridSearchCV (this may take a minute)...')
grid_search.fit(X_train, y_train)

print(f'\nBest params            : {grid_search.best_params_}')
print(f'Best CV balanced acc   : {grid_search.best_score_:.4f}')

## 8 — Evaluate on Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
print()

# ── Full classification report ────────────────────────────────────────────────
present_labels = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names   = [LABEL_NAMES.get(l, str(l)) for l in present_labels]

print('Classification Report:')
print(classification_report(
    y_test, y_pred,
    labels=present_labels,
    target_names=target_names,
    zero_division=0
))

## 9 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=present_labels)

# ── Raw counts ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disp_raw = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=target_names
)
disp_raw.plot(ax=axes[0], cmap='Blues', colorbar=False, xticks_rotation=45)
axes[0].set_title('Confusion Matrix (Raw Counts)', fontweight='bold')

# ── Normalised (row-wise recall) ──────────────────────────────────────────────
cm_norm = cm.astype(float)
row_sums = cm_norm.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1          # avoid division by zero
cm_norm = cm_norm / row_sums

disp_norm = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=target_names
)
disp_norm.plot(ax=axes[1], cmap='Blues', colorbar=False, xticks_rotation=45)
axes[1].set_title('Confusion Matrix (Row-Normalised)', fontweight='bold')

plt.suptitle('SVM Multi-Class Classification — Purkinje Cell Lick Dataset',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10 — Feature Importance (SVM Coefficient Proxy)

For RBF-kernel SVMs, direct feature weights are not available.  
We use **permutation importance** on the test set as a model-agnostic importance estimate.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=15,
    random_state=SEED,
    scoring='balanced_accuracy',
    n_jobs=-1
)

perm_imp = pd.DataFrame({
    'feature':          feature_cols,
    'importance_mean':  result.importances_mean,
    'importance_std':   result.importances_std,
}).sort_values('importance_mean', ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(4, len(feature_cols) * 0.4)))
colors = ['steelblue' if 'ss_rate' in f
          else ('tomato' if 'cs_count' in f else 'mediumseagreen')
          for f in perm_imp['feature']]

ax.barh(perm_imp['feature'], perm_imp['importance_mean'],
        xerr=perm_imp['importance_std'], color=colors,
        align='center', ecolor='grey', capsize=3)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Mean Decrease in Balanced Accuracy')
ax.set_title('Permutation Feature Importance (test set)', fontweight='bold')

# ── Legend ────────────────────────────────────────────────────────────────────
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor='steelblue',      label='SS firing rate'),
    Patch(facecolor='tomato',         label='CS spike count'),
    Patch(facecolor='mediumseagreen', label='Kinematic'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
print(perm_imp.head(5).to_string(index=False))

## 11 — Cross-Validation Score Summary

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
top_results = (
    cv_results[['param_svc__C', 'param_svc__gamma',
                'mean_test_score', 'std_test_score', 'rank_test_score']]
    .sort_values('rank_test_score')
    .head(10)
    .reset_index(drop=True)
)

print('Top 10 hyperparameter combinations (by balanced accuracy):')
print(top_results.to_string(index=False))

# ── Heatmap of mean CV score ──────────────────────────────────────────────────
pivot = cv_results.pivot_table(
    index='param_svc__C',
    columns='param_svc__gamma',
    values='mean_test_score'
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGnBu', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Balanced Accuracy'})
ax.set_title('GridSearchCV Heatmap — Balanced Accuracy', fontweight='bold')
ax.set_xlabel('gamma')
ax.set_ylabel('C')
plt.tight_layout()
plt.show()

## 12 — Per-Class Metrics Summary

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, support = precision_recall_fscore_support(
    y_test, y_pred,
    labels=present_labels,
    zero_division=0
)

metrics_df = pd.DataFrame({
    'Label':     present_labels,
    'Name':      target_names,
    'Precision': np.round(prec, 3),
    'Recall':    np.round(rec, 3),
    'F1':        np.round(f1, 3),
    'Support':   support.astype(int),
})

print(metrics_df.to_string(index=False))

# ── Bar chart ─────────────────────────────────────────────────────────────────
x = np.arange(len(target_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, prec, width, label='Precision', color='steelblue')
ax.bar(x,          rec,  width, label='Recall',    color='tomato')
ax.bar(x + width,  f1,   width, label='F1',        color='mediumseagreen')

ax.set_xticks(x)
ax.set_xticklabels(target_names, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Per-Class Precision / Recall / F1', fontweight='bold')
ax.legend()
ax.axhline(0.8, color='grey', linestyle='--', linewidth=0.8, alpha=0.6)
plt.tight_layout()
plt.show()

## 13 — Summary & Next Steps

### Results
- **Best hyperparameters**: reported above by GridSearchCV  
- **Test accuracy**: shown in Section 8  
- **Balanced accuracy (CV)**: shown in Section 11  

### Observations
- The SVM with `class_weight='balanced'` handles label imbalance (Grooming / Inter-bout dominate).
- SS firing rate features tend to carry the most discriminative signal.
- Minority classes (Right miss, Left miss, Right+reward) may show lower recall due to very few training examples.

### Potential Improvements
| Idea | Rationale |
|------|----------|
| Extend window from 100ms to 200ms | Capture more temporal context |
| Add spike-count in time bins (10ms bins) | Encode temporal dynamics, not just rate |
| SMOTE oversampling for minority classes | Improve recall on rare labels |
| Linear SVM + SelectKBest | Faster and interpretable for large feature sets |
| Replace SVM with RF or XGBoost | Non-linear models often outperform SVM on tabular neural data |
